### Google Colaboratory setup

In [1]:
#@title 0. SQAI handson environment setup

from pathlib import Path
import os
import subprocess
import sys


def running_on_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if running_on_colab():
    REPO_URL = "https://github.com/tiksato/SQAI_handson.git"
    REPO_REF = "main"
    PROJECT_ROOT = Path("/content/SQAI_handson")

    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--quiet",
                "--depth", "1",
                "--branch", REPO_REF,
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

    os.chdir(PROJECT_ROOT)

else:
    # Local case
    current = Path.cwd().resolve()

    if current.name == "notebooks":
        PROJECT_ROOT = current.parent
    else:
        PROJECT_ROOT = current

print(
    "Environment:",
    "Google Colab" if running_on_colab() else "Local Python",
)
print("Project root:", PROJECT_ROOT)

Environment: Local Python
Project root: /Users/sato/project/pyqmd_work/SQAI_handson/SQAI_notebooks


### Module import

In [2]:
import numpy as np
from pyscf import gto, scf, fci, cc

### Hartree-Fock, Coupled-Cluster, and Full-CI calculations

In [4]:
R_list = [1.0, 1.6]

basis_list = [
    'STO-3G', 
]

for R_HH in R_list:
    atoms=f'''
        H {-5*R_HH/2} 0 0
        H {-3*R_HH/2} 0 0
        H   {-R_HH/2} 0 0
        H    {R_HH/2} 0 0
        H  {3*R_HH/2} 0 0
        H  {5*R_HH/2} 0 0
        '''
    print(f"R_HH = {R_HH:.2f}")
    for basis in basis_list:
        mol = gto.M(
            atom=atoms,
            basis=basis, 
            charge=0,
            spin=0,
            symmetry=False, 
            verbose=0,
            )
        HF = scf.RHF(mol).run()
        E_HF = HF.e_tot

        mycc = cc.CCSD(HF)
        e_corr_ccsd, t1, t2 = mycc.kernel()
        E_CCSD = mycc.e_tot
        
        E_FCI, CIC = fci.FCI(HF).kernel()
        
        print(f" Number of basis: {mol.nao:3d}",
              f" HF: {E_HF:.5f}", 
              f" CCSD: {E_CCSD:.5f}", 
              f" FCI: {E_FCI:.5f}")

R_HH = 1.00
 Number of basis:   6  HF: -3.13553  CCSD: -3.23568  FCI: -3.23607
R_HH = 1.60
 Number of basis:   6  HF: -2.66498  CCSD: -2.96346  FCI: -2.95267
